In [1]:
import random
import numpy as np
import torch

def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [3]:
import pandas as pd
import torch
# from skmultilearn.model_selection import iterative_train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

2026-05-27 10:13:21.328502: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-27 10:13:21.818379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779848001.996642 3195233 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779848002.047252 3195233 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779848002.479462 3195233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [5]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [6]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [7]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [8]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [9]:
commonSubtopics

['Alternative',
 'Clinical',
 'Counseling',
 'Cultural/Religion',
 'Diagnostic Methods - Other',
 'Diet/Nutrition',
 'Emergent',
 'Exercise',
 'Financial',
 'Friends & Family',
 'Grateful Patient',
 'Health Education',
 'Hospitalizations',
 'Housing',
 'Imaging',
 'Laboratory/Testing',
 'Lifestyle - Other',
 'Medications',
 'No Symptoms',
 'Non-urgent',
 'Outpatient Logistics/Scheduling',
 'Physical',
 'Physical Environment/Climate',
 'Physical Therapy',
 'Problems Solved',
 'Procedures',
 'Request to Stop',
 'Service Complaint',
 'Social - Other',
 'Social Services',
 'Stigma Present',
 'Substance Use',
 'Technical/IT',
 'Travel',
 'Treatment(Rx) - Other',
 'Urgent',
 'Work/School']

In [10]:
dfCovid[commonSubtopics].sum()

Alternative                          35
Clinical                              5
Counseling                            0
Cultural/Religion                   154
Diagnostic Methods - Other            5
Diet/Nutrition                      102
Emergent                            138
Exercise                             25
Financial                            53
Friends & Family                    138
Grateful Patient                    244
Health Education                     45
Hospitalizations                     15
Housing                              32
Imaging                               7
Laboratory/Testing                 1018
Lifestyle - Other                    52
Medications                         152
No Symptoms                        1657
Non-urgent                         2383
Outpatient Logistics/Scheduling     268
Physical                            406
Physical Environment/Climate          6
Physical Therapy                      1
Problems Solved                      45


In [11]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]


In [12]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]


In [13]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

In [14]:
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])

In [15]:
dfCombined.sum()

conversation                       (S): Hello, how are you today? Do you have any...
Alternative                                                                       35
Clinical                                                                           6
Counseling                                                                         0
Cultural/Religion                                                                343
Diagnostic Methods - Other                                                         8
Diet/Nutrition                                                                   340
Emergent                                                                         140
Exercise                                                                         131
Financial                                                                         53
Friends & Family                                                                 291
Grateful Patient                                                 

In [16]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")

dfCombined_filtered = dfCombined[label_keep]
dfCombined_filtered.shape


(3907, 20)

In [17]:
dfCombined_filtered.sum()

conversation                       (S): Hello, how are you today? Do you have any...
Cultural/Religion                                                                343
Diet/Nutrition                                                                   340
Emergent                                                                         140
Exercise                                                                         131
Friends & Family                                                                 291
Grateful Patient                                                                 291
Health Education                                                                 322
Laboratory/Testing                                                              1078
Medications                                                                      418
No Symptoms                                                                     1658
Non-urgent                                                       

In [18]:
import numpy as np

split = np.load("data/shared_split_indices.npz")
train_idx = split["train_idx"]
val_idx   = split["val_idx"]
test_idx  = split["test_idx"]

x_raw = dfCombined_filtered["conversation"].to_numpy()
y     = dfCombined_filtered[label_keep_original].to_numpy()

x_val  = x_raw[val_idx].reshape(-1, 1)
x_test = x_raw[test_idx].reshape(-1, 1)
y_val  = y[val_idx]
y_test = y[test_idx]

df_aug_train = pd.read_csv("data/backtranslated_train.csv")
x_train = df_aug_train["conversation"]
y_train = df_aug_train[label_keep_original]

In [19]:
!pip install sentencepiece

In [20]:
from transformers import pipeline
import numpy as np

In [21]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased",do_lower_case=True)

In [22]:
train_encodings = tokenizer(x_train.tolist(), truncation=True, padding="max_length",max_length=512)
val_encodings = tokenizer(x_val.flatten().tolist(), truncation=True,  padding="max_length",max_length=512)
test_encodings = tokenizer(x_test.flatten().tolist(), truncation=True, padding="max_length",max_length=512)

In [23]:
class MultiLabelDataset(torch.utils.data.Dataset):
  def __init__(self,encodings,labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self,idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx],dtype = torch.float)
    return item
  def __len__(self):
    return len(self.labels)


In [24]:
train_dataset = MultiLabelDataset(train_encodings,y_train.to_numpy())
val_dataset = MultiLabelDataset(val_encodings,y_val)
test_dataset = MultiLabelDataset(test_encodings,y_test)

In [25]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

In [26]:
num_labels = y_train.shape[1]
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=num_labels,problem_type="multi_label_classification",hidden_dropout_prob = 0.4,attention_probs_dropout_prob = 0.4)
# y_train = y_train_new
# x_train = x_train_new

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
negativeCount = len(y_train) - y_train.sum(axis=0)
positiveCount = y_train.sum(axis=0)

weights = torch.tensor([negativeCount[i]/positiveCount[i] if positiveCount[i]>0 else 1 for i in range(len(positiveCount))],dtype=torch.float)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights = weights.to(device)

In [28]:
from torch.nn import BCEWithLogitsLoss

class WeightedLossTrainer(Trainer):
  def compute_loss(self,model,inputs,return_outputs=False, **kwargs):

    labels = inputs.pop("labels")

    outputs = model(**inputs)
    logits = outputs.get("logits")
    lossFunction = BCEWithLogitsLoss(pos_weight=weights)

    loss = lossFunction(logits, labels.float())
    return (loss,outputs) if return_outputs else loss

## Class-imbalance handling in the BERT fine-tuning loss

This BERT classifier handles severe label imbalance via **per-label positive-class weighting**
inside `BCEWithLogitsLoss`. Concretely (cell above):

```python
negativeCount = len(y_train) - y_train.sum(axis=0)
positiveCount = y_train.sum(axis=0)
weights = torch.tensor([negativeCount[i] / positiveCount[i] if positiveCount[i] > 0 else 1
                        for i in range(len(positiveCount))], dtype=torch.float)
# ...
BCEWithLogitsLoss(pos_weight=weights)
```

Each label's positive examples are up-weighted by `(# negatives) / (# positives)` in the
training set. A custom `WeightedLossTrainer` subclasses HF's `Trainer` and overrides
`compute_loss` to use this weighted BCE. No focal loss or label-smoothing is applied; no
oversampling is done in the dataloader. Thresholds are tuned per label on the validation
set (separate cell) rather than fixed at 0.5.

Reviewers asking *how* class imbalance is handled: **positive-class weighting via
`BCEWithLogitsLoss(pos_weight=N_neg/N_pos)`**, equivalent to inverse-frequency reweighting
of the positive class per label.


In [29]:
# !pip install --upgrade transformers accelerate

In [30]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import Trainer

In [31]:
# !pip install accelerate -U

In [32]:
training_args = TrainingArguments(
    output_dir='results',
    num_train_epochs=22,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='/logs',
    logging_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",

)

In [33]:
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    sigmoid_preds = 1 / (1 + np.exp(-preds))
    binary_preds = (sigmoid_preds > 0.5).astype(int)

    f1 = f1_score(y_true=p.label_ids, y_pred=binary_preds, average='macro')
    return {"f1": f1}

In [34]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


In [35]:
trainer.train()

test_results = trainer.evaluate(test_dataset)
print("\n--- Test Set Evaluation ---")
print(f"F1 Score: {test_results['eval_f1']:.4f}")

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Step,Training Loss,Validation Loss,F1
100,1.218700,1.182056,0.164930
200,1.136900,1.080273,0.282980
300,1.058400,1.032433,0.271855
400,0.979300,0.906179,0.370127
500,0.868800,0.767895,0.389942
600,0.771800,0.662731,0.422850
700,0.660400,0.611931,0.477494
800,0.608300,0.553670,0.491215
900,0.564600,0.533335,0.537918
1000,0.512800,0.513201,0.556776


/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.


--- Test Set Evaluation ---
F1 Score: 0.6939


In [36]:
# print(train_encodings)

In [37]:
best_model_path = "best_bert_WithbackTranslate"

trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print("Done Saving!")

Done Saving!


In [38]:
import numpy as np
from sklearn.metrics import f1_score

def find_optimal_threshold(trainer, eval_dataset, y_true):

    print("Finding the optimal prediction threshold...")

    predictions = trainer.predict(eval_dataset)
    logits = predictions.predictions

    sigmoid_preds = 1 / (1 + np.exp(-logits))

    best_f1 = 0
    best_threshold = 0.5 # Default

    for threshold in np.arange(0.1, 0.91, 0.01):
        binary_preds = (sigmoid_preds > threshold).astype(int)

        current_f1 = f1_score(y_true, binary_preds, average='macro', zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold

    print(f"Optimal threshold found: {best_threshold:.2f}")
    print(f"Best Macro F1 score on validation set: {best_f1:.4f}")

    return best_threshold, best_f1


optimal_threshold, best_val_f1 = find_optimal_threshold(trainer, val_dataset, y_val)

print("\n--- Evaluating on Test Set with Optimal Threshold ---")
test_predictions = trainer.predict(test_dataset)
test_logits = test_predictions.predictions
test_sigmoid_preds = 1 / (1 + np.exp(-test_logits))

final_test_preds = (test_sigmoid_preds > optimal_threshold).astype(int)

final_test_f1 = f1_score(y_test, final_test_preds, average='macro', zero_division=0)

print(f"Final Test Set Macro F1 Score: {final_test_f1:.4f}")

from sklearn.metrics import classification_report
print("\nClassification Report on Test Set:")
print(classification_report(y_test, final_test_preds, target_names=label_keep_original, zero_division=0))


Finding the optimal prediction threshold...


/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Optimal threshold found: 0.76
Best Macro F1 score on validation set: 0.7477

--- Evaluating on Test Set with Optimal Threshold ---


/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Final Test Set Macro F1 Score: 0.7079

Classification Report on Test Set:
                                 precision    recall  f1-score   support

              Cultural/Religion       0.82      0.94      0.88        53
                 Diet/Nutrition       0.77      0.94      0.85        50
                       Emergent       0.19      0.40      0.26        20
                       Exercise       0.73      0.90      0.81        21
               Friends & Family       0.68      0.70      0.69        40
               Grateful Patient       0.40      0.68      0.50        47
               Health Education       0.65      0.76      0.70        46
             Laboratory/Testing       0.93      0.97      0.95       166
                    Medications       0.89      0.89      0.89        62
                    No Symptoms       0.85      0.92      0.88       255
                     Non-urgent       0.98      0.84      0.90       522
Outpatient Logistics/Scheduling       0.75      0

In [39]:
predictions_output = trainer.predict(test_dataset)
logits = predictions_output.predictions
import torch
probs = torch.sigmoid(torch.tensor(logits)).numpy()

Y_pred_all = np.zeros_like(probs, dtype=int)
for i, label_name in enumerate(label_keep_original):
    Y_pred_all[:, i] = (probs[:, i] > optimal_threshold).astype(int)

macro_f1 = f1_score(y_test, Y_pred_all, average='macro', zero_division=0)

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [40]:
# SAVE_PREDS_v1
import numpy as np, os
os.makedirs("predictions", exist_ok=True)
np.savez_compressed(
    "predictions/BERT_preds.npz",
    y_true = y_test,
    y_pred = Y_pred_all,
)
print("Saved predictions/BERT_preds.npz", Y_pred_all.shape)


Saved predictions/BERT_preds.npz (584, 19)


In [41]:

import numpy as np
from sklearn.metrics import f1_score

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_test)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_test[idx], Y_pred_all[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)
print(f"Macro F1: {macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Macro F1: 0.7079  95% CI [0.6778, 0.7311]


## Per-corpus + per-label evaluation (BERT)

Loads saved `BERT_preds.npz` and reports Macro F1 (+ 95% bootstrap CI) and per-label F1 separately for the pooled corpus and for the Rwanda and Canada halves. Source labels come from `Ensemble_test_predictions.json`, which carries the per-row `source_corpus` field for the same 584-doc test set.


In [ ]:
# PERCORPUSCELL_v1
# Per-corpus + per-label evaluation against the shared test split, with 1,000-resample bootstrap CIs.
import json, numpy as np
from sklearn.metrics import f1_score, classification_report

with open("predictions/Ensemble_test_predictions.json") as f:
    _ens = json.load(f)
_src    = np.array([r["source_corpus"] for r in _ens])
_labels = list(_ens[0]["true_labels"].keys())

_d = np.load("predictions/BERT_preds.npz")
_yt, _yp = _d["y_true"].astype(int), _d["y_pred"]
if _yp.dtype != int:
    _yp = (_yp > 0.5).astype(int) if _yp.max() <= 1 else _yp.astype(int)

for _corpus in ["Combined", "Rwanda", "Canada"]:
    _mask = np.ones(len(_src), bool) if _corpus == "Combined" else (_src == _corpus)
    _yti, _ypi = _yt[_mask], _yp[_mask]
    _macro = f1_score(_yti, _ypi, average="macro", zero_division=0)

    _rng = np.random.default_rng(42)
    _n = len(_yti)
    _boot = np.empty(1000)
    for _i in range(1000):
        _idx = _rng.integers(0, _n, _n)
        _boot[_i] = f1_score(_yti[_idx], _ypi[_idx], average="macro", zero_division=0)
    _lo, _hi = np.percentile(_boot, [2.5, 97.5])

    print(f"\n=== {{_corpus}} (n={{_n}})  Macro F1 {{_macro:.4f}}  95% CI [{{_lo:.4f}}, {{_hi:.4f}}] ===")
    print(classification_report(_yti, _ypi, target_names=_labels, zero_division=0))
